# Pipeline PySpark cloud — Fruits! (AWS EMR)

Portage sur EMR du pipeline validé en local (`01_pipeline_local.ipynb`). La logique de traitement est strictement identique — seuls les chemins de données (S3 au lieu du disque local) et la création de la SparkSession changent.

**Prérequis avant d'exécuter ce notebook sur le cluster :**
- Cluster EMR lancé en région `eu-north-1` (même région que le bucket S3, cf. mémoire projet)
- Bootstrap action `scripts/bootstrap_emr.sh` exécutée au démarrage du cluster (installe TensorFlow + Pillow)
- Connexion établie via JupyterHub (kernel PySpark) ou EMR Notebook

## Note — SparkSession sur EMR

Contrairement à la version locale, **pas besoin de créer explicitement une `SparkSession`** ici : avec le kernel PySpark de JupyterHub (connecté via Livy au cluster), les objets `spark` et `sc` sont automatiquement disponibles dès la première cellule exécutée, déjà connectés au cluster YARN réel (nœud primaire + nœuds core/task).

À vérifier au moment de la connexion : exécuter une cellule vide ou `spark` pour confirmer que la session s'initialise (ça peut prendre quelques dizaines de secondes la première fois, le temps que YARN alloue les premiers executors).

## Définition des chemins (S3)

In [ ]:
from pyspark.sql.functions import col, element_at, split

# bucket S3 en région eu-north-1 (conforme RGPD) — pas de recherche de répertoire ici,
# contrairement au mode local : l'adresse S3 ne varie pas d'une exécution à l'autre
BUCKET = "p11-fruits-bigdata-sd2026-276005772614-eu-north-1-an"
PATH_DATA = f"s3://{BUCKET}/sample"
PATH_RESULT = f"s3://{BUCKET}/results"

print(f"PATH_DATA:   {PATH_DATA}")
print(f"PATH_RESULT: {PATH_RESULT}")

## Chargement des images au format binaire

In [ ]:
images = (
    spark.read.format("binaryFile")
    .option("pathGlobFilter", "*.jpg")
    .option("recursiveFileLookup", "true")
    .load(PATH_DATA)
)

images.printSchema()
images.select("path", "length").show(5, truncate=False)

## Extraction du label

In [ ]:
images = images.withColumn("label", element_at(split(col("path"), "/"), -2))

images.select("path", "label").show(5, truncate=False)
images.groupBy("label").count().show()

## Préparation et utilisation d'un modèle pré-entraîné (identique au local)

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2

# instanciation du modèle mobilenetv2 pré-entraîné sur ImageNet, avec la couche de sortie (top) incluse
base_model = MobileNetV2(
    weights="imagenet",
    include_top=True,
    input_shape=(224, 224, 3),
)

# reconstruction d'un modèle dont la sortie est l'avant-dernière couche (vecteur de 1280 caractéristiques)
feature_model = Model(inputs=base_model.input, outputs=base_model.layers[-2].output)
feature_model.summary()

## Broadcast des poids du modèle

In [ ]:
broadcast_weights = sc.broadcast(feature_model.get_weights())


def model_fn():
    """Rebuild the MobileNetV2 feature-extraction model and load the broadcasted weights.

    Called once per worker (not once per image), so the model is reconstructed
    only a handful of times regardless of how many images are processed.
    """
    # architecture seule : les poids sont injectés juste après via broadcast_weights
    base = MobileNetV2(weights=None, include_top=True, input_shape=(224, 224, 3))
    for layer in base.layers:
        layer.trainable = False
    model = Model(inputs=base.input, outputs=base.layers[-2].output)
    model.set_weights(broadcast_weights.value)
    return model

## Fonctions pour la chaîne de featurisation (identique au local)

In [ ]:
import io

import numpy as np
import pandas as pd
from PIL import Image
from pyspark.sql.functions import PandasUDFType, pandas_udf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array


def preprocess(content: bytes) -> np.ndarray:
    """Decode raw JPEG bytes and prepare a single image for MobileNetV2."""
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)


def featurize_series(model: Model, content_series: pd.Series) -> pd.Series:
    """Featurize a batch of raw images using the given model."""
    input_batch = np.stack(content_series.map(preprocess))
    preds = model.predict(input_batch)
    output = [p.flatten() for p in preds]
    return pd.Series(output)


@pandas_udf("array<float>", PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    """Scalar Iterator pandas UDF: loads the model once, then reuses it across batches."""
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

## Exécution

**Point à ajuster sur le cluster réel** : `repartition(8)` était calibré sur les 16 cœurs logiques de la machine locale. Sur EMR, la bonne valeur dépend du nombre d'executors réellement alloués par YARN — à vérifier via la Spark UI une fois le cluster connecté (section "Executors"), et ajuster en conséquence plutôt que de garder `8` tel quel.

In [ ]:
features_df = images.repartition(8).select(
    col("path"),
    col("label"),
    featurize_udf("content").alias("features"),
)

features_df.persist()
print(f"Nombre d'images featurisées : {features_df.count()}")
features_df.select("path", "label").show(5, truncate=False)

## Réduction de dimension (PCA PySpark)

**Point de vigilance déjà identifié en local** : le `k` choisi ici (variance expliquée à 90%) a été calibré sur l'échantillon de 10 classes non représentatif. À revalider sur ce jeu de données cloud.

In [ ]:
from pyspark.ml.feature import PCA, StandardScaler
from pyspark.ml.functions import array_to_vector, vector_to_array

vector_df = features_df.select(
    "path",
    "label",
    array_to_vector("features").alias("features_vec"),
)

scaler = StandardScaler(
    inputCol="features_vec",
    outputCol="scaled_features",
    withMean=True,
    withStd=True,
)
scaler_model = scaler.fit(vector_df)
scaled_df = scaler_model.transform(vector_df)

scaled_df.select("path", "label", "scaled_features").show(3, truncate=80)

In [ ]:
pca_explore = PCA(k=100, inputCol="scaled_features", outputCol="pca_features_full")
pca_explore_model = pca_explore.fit(scaled_df)

explained_variance = pca_explore_model.explainedVariance.toArray()
cumulative_variance = np.cumsum(explained_variance)
k_90 = int(np.argmax(cumulative_variance >= 0.90) + 1)

print(f"Composantes nécessaires pour 90% de variance expliquée : {k_90}")
print(f"Variance expliquée avec {k_90} composantes : {cumulative_variance[k_90 - 1]:.4f}")

In [ ]:
pca = PCA(k=k_90, inputCol="scaled_features", outputCol="pca_features")
pca_model = pca.fit(scaled_df)
pca_df = pca_model.transform(scaled_df)

pca_df.select("path", "label", "pca_features").show(3, truncate=80)

## Sauvegarde des résultats (S3, parquet)

In [ ]:
result_df = pca_df.select(
    "path",
    "label",
    vector_to_array("pca_features").alias("pca_features"),
)

result_df.write.mode("overwrite").parquet(PATH_RESULT)
print(f"Résultats sauvegardés dans {PATH_RESULT}")

## Validation

**Changement par rapport au local** : on relit avec Spark plutôt que `pandas.read_parquet` directement — lire un chemin S3 en pandas nécessiterait la librairie `s3fs` en plus (non installée par le bootstrap). Relire via Spark puis convertir un échantillon en pandas avec `.toPandas()` évite cette dépendance supplémentaire.

In [ ]:
validation_df = spark.read.parquet(PATH_RESULT).limit(5).toPandas()

print(validation_df.head())
print(f"\nDimension du vecteur PCA : {len(validation_df.loc[0, 'pca_features'])}")

original_dim = 1280
reduced_dim = k_90
reduction_pct = 100 * (1 - reduced_dim / original_dim)
print(f"\nRéduction de dimension : {original_dim} -> {reduced_dim} ({reduction_pct:.1f}% de moins)")